In [8]:
from typing import Any, Dict, Literal, Optional

import math
import os
import random
import sys

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["RAY_memory_monitor_refresh_ms"] = "0"

from argparse import ArgumentParser

from datetime import datetime
from timeit import default_timer as timer

import ray
from ray import tune, train as ray_train
from ray.tune.logger.aim import AimLoggerCallback
from ray.tune.logger.mlflow import MLflowLoggerCallback

import numpy as np

from tqdm import tqdm

import torch

from torch_frame.data import StatType

from torch_geometric.data import HeteroData
from torch_geometric.loader import NeighborLoader


from relbench.base import BaseTask, EntityTask, TaskType
from relbench.modeling.graph import get_node_train_table_input
from relbench.tasks import get_task

sys.path.append(".")

from redelex.tasks import CTUBaseEntityTask, CTUEntityTaskTemporal
from redelex.utils import standardize_table_dt
from redelex.nn.models.sagegnn import SAGEModel
from redelex.nn.models.dbformer import DBFormerModel

from experiments.utils import (
    get_cache_path,
    get_data,
    get_loss,
    get_metrics,
    get_tune_metric,
)

In [9]:

def get_model(architecture: Literal["sage", "dbformer"], entity_table: str, **kwargs):
    if architecture == "sage":
        return SAGEModel(**kwargs)
    elif architecture == "dbformer":
        return DBFormerModel(entity_table=entity_table, **kwargs)
    else:
        raise ValueError(f"Unknown architecture: {architecture}")


In [10]:
def run_experiment(
    data: HeteroData,
    task: BaseTask,
    dataset_name,
    task_name,
    col_stats_dict,
    model_architecture="sage",
    row_encoder="linear",
    num_layers=2,
    channels=64,
    lr=1e-3,
    batch_size=256,
    num_neighbors=32,
    min_epochs=3,
    max_steps_per_epoch=200,
    device=None,
):

    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device:", device)


    loss_fn, out_channels = get_loss(dataset_name, task_name)
    tune_metric, higher_is_better = get_tune_metric(dataset_name, task_name)
    metrics = get_metrics(dataset_name, task_name)


    loader_dict: Dict[str, NeighborLoader] = {}
    is_temporal = hasattr(task, "timestamp_col")

    for split in ["train", "val", "test"]:
        table = task.get_table(split, mask_input_cols=False)
        standardize_table_dt(table)
        table_input = get_node_train_table_input(table=table, task=task)

        loader_dict[split] = NeighborLoader(
            data,
            num_neighbors=[int(num_neighbors / (2 ** i)) for i in range(num_layers)],
            time_attr="time" if is_temporal else None,
            input_nodes=table_input.nodes,
            input_time=table_input.time if is_temporal else None,
            transform=table_input.transform,
            batch_size=batch_size,
            shuffle=(split == "train"),
        )

    model = get_model(
        architecture=model_architecture,
        entity_table=task.entity_table,
        data=data,
        col_stats_dict=col_stats_dict,
        num_layers=num_layers,
        channels=channels,
        tabular_model=row_encoder,        # row_encore / tabular_model
        out_channels=out_channels,
        aggr="sum",
        norm="batch_norm",
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    def train(split: str = "train") -> float:
        model.train()

        loader = loader_dict[split]

        loss_accum = 0
        count_accum = 0
        steps = 0

        for batch in tqdm(loader, total=min(len(loader), max_steps_per_epoch)):
            batch = batch.to(device)

            optimizer.zero_grad()
            pred = model(batch, task.entity_table)
            pred = pred.view(-1) if pred.size(1) == 1 else pred

            if pred.size(0) != batch[task.entity_table].batch_size:
                pred = pred[: batch[task.entity_table].batch_size]

            if task.task_type == TaskType.MULTICLASS_CLASSIFICATION:
                target = batch[task.entity_table].y.long()
            else:
                target = batch[task.entity_table].y.float()

            loss = loss_fn(pred.float(), target)
            loss.backward()
            optimizer.step()

            loss_accum += loss.detach().item() * pred.size(0)
            count_accum += pred.size(0)

            steps += 1
            if steps >= max_steps_per_epoch:
                break

        return loss_accum / count_accum

    @torch.no_grad()
    def evaluate(split):
        model.eval()
        loader = loader_dict[split]

        pred_list = []
        for batch in tqdm(loader):
            batch = batch.to(device)
            pred = model(batch, task.entity_table)

            if task.task_type in [
                TaskType.BINARY_CLASSIFICATION,
                TaskType.MULTILABEL_CLASSIFICATION,
            ]:
                pred = torch.sigmoid(pred)

            if task.task_type == TaskType.MULTICLASS_CLASSIFICATION:
                pred = torch.softmax(pred, dim=1)

            pred = pred.view(-1) if pred.size(1) == 1 else pred

            if pred.size(0) != batch[task.entity_table].batch_size:
                pred = pred[: batch[task.entity_table].batch_size]

            pred_list.append(pred.detach().cpu())

        pred_list = torch.cat(pred_list).numpy()
        table = task.get_table(split)
        return task.evaluate(pred_list, table, metrics=metrics)

    for epoch in range(1, min_epochs + 1):
        print(f"\nEpoch {epoch}/{min_epochs}")
        train_loss = train()
        val_metrics = evaluate("val")

        print(f"Train loss: {train_loss:.4f}")
        print("Val metrics:", val_metrics)


    return model, val_metrics


In [11]:
from relbench.tasks import get_task
from experiments.utils import get_data, get_cache_path

dataset_name = "rel-f1"
task_name = "driver-position"

cache_path = get_cache_path(dataset_name, task_name, ".cache")
task, data, col_stats_dict = get_data(dataset_name, task_name, cache_path)

model, val_m = run_experiment(
    data=data,
    task=task,
    dataset_name=dataset_name,
    task_name=task_name,
    col_stats_dict=col_stats_dict,
    model_architecture="sage",
    row_encoder="linear",
    num_layers=2,
    channels=64,
    lr=1e-3,
    batch_size=256,
    num_neighbors=32,
    min_epochs=10,
    max_steps_per_epoch=200,
)


/home/gabrimi8/RDL/ReDeLEx/.venv/lib/python3.12/site-packages/torch_frame/utils/io.py:113: UserWarning: Weights only load failed. Please file an issue to make `torch.load(weights_only=True)` compatible in your case. Please use `torch.serialization.add_safe_globals([scalar])` to allowlist this global.
  warnings.warn(f"{warn_msg} Please use "
/home/gabrimi8/RDL/ReDeLEx/.venv/lib/python3.12/site-packages/torch_frame/utils/io.py:113: UserWarning: Weights only load failed. Please file an issue to make `torch.load(weights_only=True)` compatible in your case. Please use `torch.serialization.add_safe_globals([scalar])` to allowlist this global.
  warnings.warn(f"{warn_msg} Please use "
/home/gabrimi8/RDL/ReDeLEx/.venv/lib/python3.12/site-packages/torch_frame/utils/io.py:113: UserWarning: Weights only load failed. Please file an issue to make `torch.load(weights_only=True)` compatible in your case. Please use `torch.serialization.add_safe_globals([scalar])` to allowlist this global.
  warnings

Device: cpu

Epoch 1/10


100%|██████████| 2/2 [00:00<00:00,  3.14it/s]


Train loss: 11.3069
Val metrics: {'mae': 7.721226502165607, 'mse': 78.9394814988348, 'r2': -2.67276964363098}

Epoch 2/10


100%|██████████| 2/2 [00:00<00:00,  4.21it/s]


...

Train loss: 5.6990
Val metrics: {'mae': 3.8778502405048134, 'mse': 22.34446443101202, 'r2': -0.03960741959806868}

Epoch 9/10


100%|██████████| 2/2 [00:00<00:00, 13.97it/s]


Train loss: 5.5589
Val metrics: {'mae': 3.5533343396030745, 'mse': 18.21580011627689, 'r2': 0.15248445478453065}

Epoch 10/10


100%|██████████| 2/2 [00:00<00:00,  6.40it/s]

Train loss: 5.2600
Val metrics: {'mae': 3.531036839137972, 'mse': 18.823328242862555, 'r2': 0.12421836006729114}


In [6]:
print(val_m)

{'mae': 3.4683919361295428, 'mse': 18.414432403067934, 'r2': 0.14324281018135532}


In [1]:
from relbench.tasks import get_task, get_task_names
get_task_names("rel-f1")
get_task("rel-f1", "driver-position")

DriverPositionTask(dataset=F1Dataset())

In [16]:
from relbench.datasets import get_dataset, get_dataset_names

from redelex.datasets import ErgastF1
from redelex.db import DBInspector
from redelex.db.utils import get_rdb_connection

print(get_dataset_names())
dataset = get_dataset("ctu-airline")


['rel-amazon', 'rel-avito', 'rel-event', 'rel-f1', 'rel-hm', 'rel-stack', 'rel-trial', 'ctu-accidents', 'ctu-adventureworks', 'ctu-airline', 'ctu-atherosclerosis', 'ctu-basketballmen', 'ctu-basketballwomen', 'ctu-biodegradability', 'ctu-bupa', 'ctu-carcinogenesis', 'ctu-cde', 'ctu-chess', 'ctu-classicmodels', 'ctu-cora', 'ctu-countries', 'ctu-craftbeer', 'ctu-credit', 'ctu-dallas', 'ctu-dcg', 'ctu-diabetes', 'ctu-dunur', 'ctu-elti', 'ctu-employee', 'ctu-ergastf1', 'ctu-expenditures', 'ctu-financial', 'ctu-fnhk', 'ctu-ftp', 'ctu-geneea', 'ctu-genes', 'ctu-gosales', 'ctu-grants', 'ctu-hepatitis', 'ctu-hockey', 'ctu-imdb', 'ctu-lahman', 'ctu-legalacts', 'ctu-mesh', 'ctu-mondial', 'ctu-mooney', 'ctu-movielens', 'ctu-musklarge', 'ctu-musksmall', 'ctu-mutagenesis', 'ctu-ncaa', 'ctu-northwind', 'ctu-pima', 'ctu-premiereleague', 'ctu-restbase', 'ctu-sakila', 'ctu-sales', 'ctu-samegen', 'ctu-sap', 'ctu-satellite', 'ctu-seznam', 'ctu-sfscores', 'ctu-shakespeare', 'ctu-stats', 'ctu-studentloan', 

In [17]:
inspector = DBInspector(get_rdb_connection(dataset.remote_url))

for t in inspector.get_tables():
    print(inspector.get_foreign_keys(t))
inspector.connection.close()

[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[ForeignKey(src_columns=['AirlineID'], ref_table='L_AIRLINE_ID', ref_columns=['Code']), ForeignKey(src_columns=['ArrDel15'], ref_table='L_YESNO_RESP', ref_columns=['Code']), ForeignKey(src_columns=['ArrivalDelayGroups'], ref_table='L_ONTIME_DELAY_GROUPS', ref_columns=['Code']), ForeignKey(src_columns=['CancellationCode'], ref_table='L_CANCELLATION', ref_columns=['Code']), ForeignKey(src_columns=['Cancelled'], ref_table='L_YESNO_RESP', ref_columns=['Code']), ForeignKey(src_columns=['DayOfWeek'], ref_table='L_WEEKDAYS', ref_columns=['Code']), ForeignKey(src_columns=['DepDel15'], ref_table='L_YESNO_RESP', ref_columns=['Code']), ForeignKey(src_columns=['DepTimeBlk'], ref_table='L_DEPARRBLK', ref_columns=['Code']), ForeignKey(src_columns=['DepartureDelayGroups'], ref_table='L_ONTIME_DELAY_GROUPS', ref_columns=['Code']), ForeignKey(src_columns=['DestAirportID'], ref_table='L_AIRPORT_ID', ref_columns=['Code']), ForeignKey(src_columns=['DestAirportSeqID'], r

In [1]:
import torch
from experiments.utils import test_get_data
import importlib
import redelex.data as rgraph

importlib.reload(rgraph)
from experiments.utils import get_cache_path, GloveTextEmbedding
from torch_frame.config.text_embedder import TextEmbedderConfig

cache_dir = ".cache"
dataset_name = "rel-f1"
task_name = "driver-position"
cache_path = get_cache_path(dataset_name, task_name, cache_dir)
db, attribute_schema = test_get_data(dataset_name, task_name, cache_path)
#, col_stats_dict
data, col = rgraph.make_pkey_fkey_graph(db,
        col_to_stype_dict=attribute_schema,
        text_embedder_cfg=TextEmbedderConfig(
            text_embedder=GloveTextEmbedding(device=torch.device("cpu")), batch_size=256
        ),
        cache_dir=f"{cache_path}/materialized",
    )
print(data.tensor_frame)
print("XXX")
print(data.col_stats)
print("XXX")
print(col)


Loading Database object from /home/gabrimi8/.cache/relbench/rel-f1/db...
Done in 0.14 seconds.
TensorFrame(
  num_cols=2,
  num_rows=12290,
  numerical (1): ['points'],
  timestamp (1): ['date'],
  has_target=False,
  device='cpu',
)
XXX
{'points': {<StatType.MEAN: 'MEAN'>: 3.859967453213995, <StatType.STD: 'STD'>: 7.554828092086204, <StatType.QUANTILES: 'QUANTILES'>: [0.0, 0.0, 0.0, 4.0, 66.0]}, 'date': {<StatType.YEAR_RANGE: 'YEAR_RANGE'>: [np.int32(1956), np.int32(2023)], <StatType.NEWEST_TIME: 'NEWEST_TIME'>: tensor([2023,    6,   29,    6,   13,    0,    0]), <StatType.OLDEST_TIME: 'OLDEST_TIME'>: tensor([1956,    0,   21,    6,    0,    0,    0]), <StatType.MEDIAN_TIME: 'MEDIAN_TIME'>: tensor([1992,    6,   25,    6,    0,    0,    0])}}
XXX
{'constructor_results': {'points': {<StatType.MEAN: 'MEAN'>: 3.859967453213995, <StatType.STD: 'STD'>: 7.554828092086204, <StatType.QUANTILES: 'QUANTILES'>: [0.0, 0.0, 0.0, 4.0, 66.0]}, 'date': {<StatType.YEAR_RANGE: 'YEAR_RANGE'>: [np.int32(

/home/gabrimi8/RDL/ReDeLEx/.venv/lib/python3.12/site-packages/torch_frame/utils/io.py:113: UserWarning: Weights only load failed. Please file an issue to make `torch.load(weights_only=True)` compatible in your case. Please use `torch.serialization.add_safe_globals([scalar])` to allowlist this global.
  warnings.warn(f"{warn_msg} Please use "
